# 07 — Experimento 1: atributos de ocorrência e intensidade (seção 24 do v5)

**Lê:** treino, `panel_base` até `TRAIN_TARGET_END`, `painel` até `CV_FIRST_ORIGIN` (correlações), `selecao_modelo.json`.  
**Grava:** `limites_prototipo.json` (congelado antes da validação), resultados por origem/perfil, `experimento1_modelo_escolhido.json`. O teste só é carregado se alguma variante for escolhida.

**Status para o texto:** a ablação é feita sobre o `CatBoost_MAE` (mesma perda e hiperparâmetros), não sobre o modelo escolhido na seção 16. Como as hipóteses (ocorrência, picos) surgiram do diagnóstico do teste (notebook 06), um eventual resultado no teste é **exploratório**, não confirmatório.

In [1]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "07_experimento1"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

AVISO de reprodutibilidade: pandas=3.1.1; referência recomendada=2.2.3
AVISO de reprodutibilidade: numpy=2.5.4; referência recomendada=2.1.3
AVISO de reprodutibilidade: scikit-learn=1.9.1; referência recomendada=1.6.1
Notebook: 07_experimento1 | execução: v6_modular | raiz: C:\Users\andre\OneDrive\Documentos\Projetos\Tccv2\tcc_fluxo_modular_v6


In [2]:
from tcc_fluxo import experimento1 as E, cv, Especificacao
from tcc_fluxo.metricas import seller_scales

spec = Especificacao.de_json(A.carregar_json("painel_spec"), C)
sel = A.carregar_json("selecao_modelo")
train = A.carregar_treino()
hist = A.carregar_historico_treino()
cv_origins, _ = cv.origens_cv(C)
assert len(cv_origins) == 6

extra = E.build_extra_features(hist)                     # causal: idêntico ao cálculo sobre a série completa
print(f"{len(E.EXP1_ALL)} atributos novos para {len(extra):,} linhas")

63 atributos novos para 32,277 linhas


### 24.1 Verificações dos atributos (vazamento, redundâncias, exemplo conceitual)

In [3]:
rng = np.random.default_rng(C.RANDOM_SEED)
pert = hist.copy()
fut = pert["week_start"] > C.LEAK_TEST_CUT
pert.loc[fut, "revenue"] = pert.loc[fut, "revenue"] * rng.uniform(0.5, 3.0, fut.sum()) + 1000.0
pert.loc[fut, "orders"] = pert.loc[fut, "orders"] + 7
changed, check = E.verificar_atributos(extra, E.build_extra_features(pert), C.LEAK_TEST_CUT, hist)
print(f"Sem vazamento: OK (controle: {changed:.0%} dos atributos mudam depois do corte) | exemplo: {check}")

extra_cols = extra.drop(columns="zero_streak")
train_x = train.merge(extra_cols, on=["seller_id", "week_start"], how="left", validate="one_to_one")
assert len(train_x) == len(train)

Sem vazamento: OK (controle: 100% dos atributos mudam depois do corte) | exemplo: {'n_pos_8': np.float64(3.0), 'freq_8': np.float64(0.375), 'pos_mean_8': np.float64(83.33333333333333), 'pos_median_8': np.float64(80.0), 'max_8': np.float64(120.0), 'weeks_since_last_sale': np.float64(0.0)}


### 24.2 Decisões congeladas antes da validação

In [4]:
assert np.allclose(train_x["pos_sum_4"], train_x[["revenue_current", "rev_lag_1", "rev_lag_2", "rev_lag_3"]].sum(axis=1))
assert np.allclose(train_x["max_4"], train_x[["revenue_current", "rev_lag_1", "rev_lag_2", "rev_lag_3"]].max(axis=1))
corr_df = (hist.loc[hist["week_start"] <= C.CV_FIRST_ORIGIN, ["seller_id", "week_start"]]
           .merge(A.carregar_painel(C.CV_FIRST_ORIGIN)[["seller_id", "week_start"] + spec.numeric],
                  on=["seller_id", "week_start"])
           .merge(extra_cols, on=["seller_id", "week_start"]))
INT_RED, l1 = E.reduce_block(E.EXP1_BLOCKS["intensidade"], "intensidade_red", corr_df, spec.numeric, C.CORR_MAX)
PIC_RED, l2 = E.reduce_block(E.EXP1_BLOCKS["picos"], "picos_red", corr_df, spec.numeric, C.CORR_MAX)
INTPIC_RED, l3 = E.reduce_block(INT_RED + PIC_RED, "intensidade_picos_red", corr_df, spec.numeric, C.CORR_MAX)
A.tabela(pd.DataFrame(l1 + l2 + l3), "experimento1_reducao_atributos.csv")

LIMITS, cal_df = E.calibrar_limites(train, hist, C)
A.tabela(cal_df, "limites_calibracao_pre_cv.csv"); display(cal_df)
ROUNDS = E.rodadas(INT_RED, PIC_RED, INTPIC_RED, spec.exog)
frozen, LIMITS_HASH = E.congelar_decisoes(LIMITS, ROUNDS, cv_origins, C)
A.salvar_json(frozen, "limites_prototipo")
print(f"Decisões congeladas; sha256={LIMITS_HASH[:16]}…")

,perfil,MAE,MAE_positiva,P90,P95,vies,real_medio,n_linhas,n_vendedores,limite_MAE,limite_MAE_positiva,limite_P90,limite_P95,limite_vies_abs_max
0,intermitente,74.4984,269.3857,174.2630,339.6845,-1.3642,42.3994,1798,193,81.9482,296.3242,191.6893,373.6529,4.6639
1,medio,173.4670,261.5382,402.4300,678.2020,1.1068,120.7180,1799,169,190.8138,287.6921,442.6730,746.0222,13.2790
2,regular,334.4017,350.2454,754.3700,"1,164.2675",-2.0838,471.1255,1946,175,367.8419,385.2700,829.8070,"1,280.6943",51.8238


Decisões congeladas; sha256=369bbeab4ecbd90a…


### 24.3 Validação nas seis origens

In [5]:
perfil_val, exp1_long, EXP1_MODELS = E.validar(train_x, hist, cv_origins, ROUNDS, spec, C)
A.salvar_df(perfil_val, "experimento1_perfil_validacao")
A.tabela(perfil_val, "experimento1_perfil_validacao.csv")
cv_detail = A.carregar_df("cv_detalhe")
ref_cv = cv_detail.loc[cv_detail["modelo"] == "CatBoost_MAE", "MASE_vend_media"].to_numpy()
base_cv = perfil_val.query("rodada == 'base' and perfil == 'todos'")["MASE_vend_media"].to_numpy()
assert np.allclose(ref_cv, base_cv), (ref_cv, base_cv)
print("Rodada 'base' reproduz exatamente o CatBoost_MAE da validação principal: OK")

origem 2017-06-26 | treino  5,543 | val 2,311 | perfis {'intermitente': 209, 'medio': 196, 'regular': 195} |   418s
origem 2017-07-24 | treino  7,854 | val 2,496 | perfis {'intermitente': 233, 'medio': 215, 'regular': 188} |   849s
origem 2017-08-21 | treino 10,350 | val 2,706 | perfis {'intermitente': 255, 'medio': 232, 'regular': 220} |  1267s
origem 2017-09-18 | treino 13,056 | val 3,001 | perfis {'intermitente': 274, 'medio': 252, 'regular': 248} |  1690s
origem 2017-10-16 | treino 16,057 | val 3,234 | perfis {'intermitente': 287, 'medio': 278, 'regular': 268} |  2126s
origem 2017-11-13 | treino 19,291 | val 3,416 | perfis {'intermitente': 290, 'medio': 290, 'regular': 290} |  2574s
Rodada 'base' reproduz exatamente o CatBoost_MAE da validação principal: OK


### 24.4 Ganho por perfil

In [6]:
prof_mean, prof_delta = E.ganho_por_perfil(perfil_val, ROUNDS)
A.tabela(prof_delta, "experimento1_ganho_por_perfil.csv")
display(prof_delta.set_index(["rodada", "perfil"]))
display(E.perguntas(prof_delta))

Δ MAE  Δ MAE_positiva  Δ MAE_zero    Δ P95  Δ vies  Δ picos_previsto_real  Δ previsto_medio  Δ pct_previsao_quase_zero  Δ MASE %  alerta_quase_zero
rodada                  perfil                                                                                                                                                            
+intensidade            todos        -1.6268         -5.2063      1.0182   3.7329  7.6128                 0.0440            7.6128                    -0.2832   -0.3755              False
                        intermitente -0.0422         -0.3119      0.0181   0.0294  0.0674                -0.0005            0.0674                    -0.0186   -0.1101              False
                        medio        -0.5039         -2.1679      0.4703  -1.7308  2.4601                 0.0505            2.4601                    -0.8387   -0.6497              False
                        regular      -4.5537         -8.0773      6.5056 -29.3386 21.6269                 0.0457           21.6269                    -0.0029   -0.4540              False
+picos                  todos        -2.3190         -6.9625      1.1863  -6.2522  6.0434                 0.0335            6.0434                    -0.2117   -0.4398              False
                        intermitente -0.0184         -0.2577      0.0441   0.4292  0.0842                -0.0002            0.0842                    -0.0716   -0.3549              False
                        medio        -0.1024         -1.0452      0.4921  -1.9150  1.7758                 0.0280            1.7758                    -0.4309   -0.2971              False
                        regular      -7.2541        -11.9433      7.3819 -27.4474 17.2820                 0.0360           17.2820                    -0.1255   -0.6979              False
+todos                  todos        -2.2346         -5.6906      0.3824  -8.4628  3.6152                 0.0139            3.6152                    -0.3480   -0.9956              False
                        intermitente -0.0632          0.1354     -0.1256  -0.7206 -0.1415                -0.0010           -0.1415                     0.0238   -0.5160              False
                        medio        -0.7084         -0.5145     -0.7748   1.2631  0.2345                 0.0159            0.2345                     0.0176   -1.1536              False
                        regular      -6.2131         -9.9942      5.1379 -35.3306 11.4727                 0.0144           11.4727                    -1.1087   -1.4443              False
+pedidos                todos        -1.2516         -2.8534      0.0696   0.6080  1.5753                 0.0071            1.5753                    -0.6192   -0.9154              False
                        intermitente -0.1439         -0.3397     -0.1035   0.0754 -0.0097                -0.0008           -0.0097                    -0.0515   -0.6796              False
                        medio        -1.0405         -2.8081      0.0906  -0.5237  2.0857                 0.0261            2.0857                    -0.9044   -1.1672              False
                        regular      -2.6770         -3.6202      0.7061 -14.1457  3.0576                 0.0066            3.0576                    -0.9250   -0.9378              False
+intensidade_red        todos        -1.7732         -5.3909      0.8154 -11.1937  5.5512                 0.0377            5.5512                    -0.0694   -0.5218              False
                        intermitente -0.0088         -0.1345      0.0192  -1.3183  0.0364                -0.0004            0.0364                     0.0215   -0.3262              False
                        medio        -0.4338         -1.6627      0.2292  -0.5257  1.7137                 0.0364            1.7137                    -0.2458   -0.6022              False
                        regular      -5.1633         -8.7571      5.3229 -20.7947 15.9611                 0.0397

,pergunta,Δ MAE,Δ MAE positiva,resposta,Δ previsto/real
0,+intensidade melhorou os intermitentes?,-0.0422,-0.3119,sim,NaN
1,+intensidade melhorou os médios?,-0.5039,-2.1679,sim,NaN
2,+intensidade melhorou os regulares?,-4.5537,-8.0773,sim,NaN
3,+intensidade_red melhorou os intermitentes?,-0.0088,-0.1345,sim,NaN
4,+intensidade_red melhorou os médios?,-0.4338,-1.6627,sim,NaN
5,+intensidade_red melhorou os regulares?,-5.1633,-8.7571,sim,NaN
6,+picos melhora os picos dos regulares?,NaN,-11.9433,sim,0.0360
7,+picos_red melhora os picos dos regulares?,NaN,-8.2007,sim,0.0249
8,Algum bloco reduz o MASE só por prever perto d...,NaN,NaN,não,NaN


### 24.5 Critérios por perfil e conformidade individual

In [7]:
seller_c, comp_o, comp_s, crit_perfil, limits_all = E.conformidade(exp1_long, perfil_val, prof_mean, ROUNDS,
                                                                    cv_origins, LIMITS, C)
A.tabela(seller_c, "experimento1_conformidade_vendedor.csv")
A.tabela(comp_s, "experimento1_conformidade_vendedor_resumo.csv")
A.tabela(crit_perfil, "experimento1_criterios_por_perfil.csv")
display(crit_perfil[["rodada", "perfil", "MAE_ok", "MAE_positiva_ok", "P90_ok", "P95_ok", "vies_abs_ok",
                     "origens_dentro_dos_limites", "pct_vendedores_dentro_limites_media", "passou"]])
print("Passa nos três perfis:", limits_all.to_dict())

,rodada,perfil,MAE_ok,MAE_positiva_ok,P90_ok,P95_ok,vies_abs_ok,origens_dentro_dos_limites,pct_vendedores_dentro_limites_media,passou
0,base,intermitente,True,True,True,True,False,0,0.5401,False
1,base,medio,True,True,True,True,False,0,0.3308,False
2,base,regular,True,True,True,True,False,0,0.3686,False
3,+intensidade,intermitente,True,True,True,True,False,0,0.5394,False
4,+intensidade,medio,True,True,True,True,False,0,0.3317,False
5,+intensidade,regular,True,True,True,True,False,0,0.3734,False
6,+picos,intermitente,True,True,True,True,False,0,0.5394,False
7,+picos,medio,True,True,True,True,False,0,0.3305,False
8,+picos,regular,True,True,True,True,False,0,0.3580,False
9,+todos,intermitente,True,True,True,True,False,0,0.5427,False


Passa nos três perfis: {'base': False, '+intensidade': False, '+picos': False, '+todos': False, '+pedidos': False, '+intensidade_red': False, '+picos_red': False, '+intensidade_picos_red': False, '+macro_calendario': False, '+todos_macro_calendario': False}


### 24.6 Critérios globais, de picos e bootstrap das diferenças

In [8]:
boot_df, glob_crit, glob, glob_wape = E.criterios_globais(exp1_long, perfil_val, prof_mean, ROUNDS, C)
A.tabela(boot_df, "experimento1_diferencas_bootstrap.csv")
A.tabela(glob_crit, "experimento1_criterios_globais.csv", index=True)
display(glob_crit)

,G1_MASE,G2_MAE_positiva,G3_falso_sinal,G4_WAPE_agregado,G5_origens,G6_robustez,P1_delta_min,P2_origens,P3_regulares,P4_ic95,globais_ok,picos_ok,delta_MASE_pct,delta_MAE_positiva,delta_falso_sinal_pp,delta_WAPE_agregado_pct,delta_picos_previsto_real,ic95_delta_picos,origens_MASE_melhor,origens_picos_melhor,pct_regulares_picos_melhor,n_regulares_com_pico,pct_semanas_melhor,pct_vendedores_melhor,bootstrap_estimando
rodada,,,,,,,,,,,,,,,,,,,,,,,,,
+intensidade,True,True,True,True,True,True,True,True,True,True,True,True,-0.3755,-5.2063,0.1096,-8.6067,0.0440,[+0.020; +0.063],6,6,67.2131,122,66.6667,52.5892,média simples entre origens
+picos,True,True,True,True,True,True,True,True,True,True,True,True,-0.4398,-6.9625,0.0992,-6.6342,0.0335,[+0.016; +0.048],5,6,71.3115,122,83.3333,56.6168,média simples entre origens
+todos,True,True,True,True,True,True,False,True,True,True,True,False,-0.9956,-5.6906,-0.1180,-3.9843,0.0139,[+0.003; +0.027],6,4,59.0164,122,79.1667,64.4752,média simples entre origens
+pedidos,True,True,True,True,True,True,False,True,True,False,True,False,-0.9154,-2.8534,-0.0151,-1.5580,0.0071,[-0.011; +0.027],6,4,52.4590,122,66.6667,70.0806,média simples entre origens
+intensidade_red,True,True,True,True,True,True,True,True,True,True,True,True,-0.5218,-5.3909,0.0912,-6.3128,0.0377,[+0.014; +0.057],6,6,65.5738,122,70.8333,57.9493,média simples entre origens
+picos_red,True,True,True,True,True,True,True,True,True,True,True,True,-0.2295,-4.4828,0.1219,-5.1472,0.0231,[+0.009; +0.037],5,4,68.8525,122,75.0000,51.0369,média simples entre origens
+intensidade_picos_red,True,True,True,True,True,True,True,True,True,False,True,False,-0.4035,-5.2967,-0.1825,-4.3187,0.0206,[-0.001; +0.037],5,4,59.0164,122,79.1667,58.2759,média simples entre origens
+macro_calendario,True,True,True,True,True,True,False,False,False,False,True,False,-0.1928,-2.7918,-0.2187,0.1133,0.0018,[-0.010; +0.012],5,3,44.2623,122,79.1667,53.6332,média simples entre origens
+todos_macro_calendario,True,True,True,True,True,True,False,True,True,False,True,False,-1.2958,-5.4592,-0.1095,-4.1099,0.0107,[-0.002; +0.024],6,4,68.0328,122,75.0000,66.3982,média simples entre origens


### 24.7 Seleção (somente validação)

In [9]:
frozen_now = A.carregar_json("limites_prototipo")
assert E.hash_decisoes(frozen_now) == LIMITS_HASH, "limites alterados após o congelamento"
elig, exp1_choice = E.selecionar(ROUNDS, glob, glob_crit, limits_all, crit_perfil, LIMITS_HASH, cv_origins, C)
exp1_choice["modelo_principal_secao16"] = sel["modelo_escolhido"]
A.salvar_json(exp1_choice, "experimento1_modelo_escolhido")
display(elig)
print(exp1_choice["decisao"])
sens, el, tied, hip = E.sensibilidade_sem_limites(ROUNDS, glob, glob_crit, C)
print(f"[informativo] sem os limites por perfil: elegíveis {el}; empate {tied}; escolha hipotética {hip}")

,n_atributos,MASE_vend_media,limites_3_perfis,globais,picos,elegivel
rodada,,,,,,
base,17,1.0005,False,n/a,n/a,False
+intensidade,33,0.9968,False,True,True,False
+picos,29,0.9961,False,True,True,False
+todos,80,0.9905,False,True,False,False
+pedidos,26,0.9914,False,True,False,False
+intensidade_red,27,0.9953,False,True,True,False
+picos_red,24,0.9982,False,True,True,False
+intensidade_picos_red,33,0.9965,False,True,False,False
+macro_calendario,43,0.9986,False,True,False,False


nenhum candidato passou nos limites dos três perfis e nos demais critérios; o Experimento 1 não escolhe modelo e o modelo principal de selecao_modelo.json é mantido
[informativo] sem os limites por perfil: elegíveis ['base', '+intensidade', '+picos', '+intensidade_red', '+picos_red']; empate ['base', '+intensidade', '+picos', '+intensidade_red', '+picos_red']; escolha hipotética base


### 24.8 Teste — somente se a validação escolheu uma variante

In [10]:
EXP1_CHOSEN = exp1_choice["modelo_escolhido"]
if EXP1_CHOSEN in (None, "base"):
    print("Teste do Experimento 1 NÃO executado:", "nenhum candidato elegível" if EXP1_CHOSEN is None else "a validação manteve a base")
else:
    from tcc_fluxo import diagnostico as D, metricas as M
    test = A.carregar_teste(finalidade="exploratoria")
    serie = A.carregar_serie_completa(finalidade="exploratoria")
    extra_full = E.build_extra_features(serie).drop(columns="zero_streak")
    train_x2 = train.merge(extra_full, on=["seller_id", "week_start"], how="left", validate="one_to_one")
    test_x = test.merge(extra_full, on=["seller_id", "week_start"], how="left", validate="one_to_one")
    comparison = A.carregar_df("previsoes_teste")
    profile = A.carregar_df("perfis_treino").set_index("seller_id")
    exp1_test, resumo_t = E.testar_escolhido(EXP1_CHOSEN, ROUNDS, train_x2, test_x, comparison, profile,
                                             M.seller_scales(hist, C.TRAIN_TARGET_END), D.limiares(hist), LIMITS, spec, C)
    A.tabela(exp1_test, "experimento1_teste.csv"); display(exp1_test)
    print("Diferença de MAE no teste (nova − base):", resumo_t)

Teste do Experimento 1 NÃO executado: nenhum candidato elegível


In [11]:
A.finalizar()

07_experimento1: leu 6 artefato(s), gravou 3.
